<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.1

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate


**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for entire course:** Murphy, Kevin P. *Probabilistic machine learning: an
introduction*. MIT press, 2022. Available online [here](https://probml.github.io/pml-book/book1.html)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the first of two lectures on **adda**. Today: how you build such a framework, and what
we learned building it. Lecture 19.2: what happened when we pointed it at a real problem.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

In [2]:
# In Google Colab you need to install f3dasm first (locally it is already in the '3dasm'
# environment). Uncomment the line below if you are running in Colab:

# %pip install f3dasm

from f3dasm import ExperimentData   # the same object you used in Lectures 17, 18 and 19

## Outline for today

* An agent, and a graph of them
* Three reasons to split the work: safety, specialization, efficiency
* The record as the state
* Admitting a claim: refutation, then reproduction
* Getting started, and one lesson from building it

**Reading material**: this notebook + the
[a3dasm documentation](https://elvis-aguero.github.io/a3dasm/).

adda is the framework: agentic data-driven design and analysis. a3dasm is the importable
package, and it is what the documentation link points at. The codebase uses a3dasm throughout.

The package and its documentation are public. The campaign shown in 19.2 is not yet published.

This lecture is the builder's view of the architecture and the decisions behind it. 19.2 applies
the same machinery to a metamaterial design problem that is also the course's final project.

## Agents, tools, and jobs


* An LLM with a set of tools, and a job is an **agent**, or a **subagent** when another agent spawns it. As of 2026 they are dispatched automatically by the coding agents you already use:

<img src=../figures/agent_products.svg width=86%>

* A fixed code path that calls a model at each step is a **workflow**. We will see how to **orchestrate** multiple of them.

Unlike plain agents, we are able to customize what each agent tools are at building time. Its generated from the live tool set and
appended to the end of the system prompt at every invocation, in _src/backends/claude.py.

Subagent means an agent that another agent started. Claude Code and competitors can already do that off-the-shelf, but they offer little customization.

## Anatomy of an agent

<p align="center"><img src=../figures/agent_anatomy.png width=64%></p>

A system prompt sets the persona. A frozen set of tools sets what it can reach. The backend is
which model runs it, and the report format is the shape it must hand back.

Claude Code and its competitors ship one general-purpose agent with a fixed tool set. Here every
node is built with a different one: the critic gets Read, Glob and Grep, the implementer gets Bash
and Write, and neither can acquire the other's at runtime.

The prompt excerpt is real charter text. Registering a falsifiable prediction before looking at
the data is the first of six numbered clauses in a3dasm/_src/knowledge/charter.py.

Four backends are registered: claude, ollama, openrouter and vllm. Two of them run locally.

## Multi-agent architectures

Multiple agents can be arranged in several ways: a single agent with tools, a network, a supervisor, a supervisor-as-tool, a hierarchy, or a custom graph.

In a graph architecture each node is a possibly different agent, and the edges say which node can interact with which.

<p align="center"><img src=../figures/agentic_patterns.png width=58%></p>

<sub>Taxonomy from the LangGraph multi-agent documentation; figure from *Agentic patterns: architectures for coordinated AI systems*, Medium.</sub>

We show the decisions involved in architecting a _graph_ of agents.

The six panels are an external taxonomy, reproduced for orientation.

a3dasm's own topology is none of them exactly. _src/agents/_graphs.py defines five nodes and six
directed edges. An agent is ultimately composed of two elements: its system prompt, and the MCP (model context protocol) tools its given. Everything else sits on top of that. The system prompt sets the personna of the agent, the mcp tools (which are to be listed in the system prompt) gives them access to the external world. 

## The case for a graph

Three principles: safety, specialization, efficiency.

## Safety I: independence needs a fresh context

An agent asked to check its own work is usually biased to be agreeable.

A second agent, started fresh and shown only the result, gives you a real second opinion.

A second model shares the biases of the first. Three things still separate the critic's verdict
from self-review: (a) It starts from a fresh context (b) It sees the artifact rather than the
reasoning that produced it, and (c) it acts as an adversarial agent.

## Safety II: restricted tools per node

Each node gets only the tools its job needs.

Every agent class declares a frozen set of tool names, and the catalog at the end of its prompt
is built from exactly that set.

The critic's set is read-only: Read, Glob and Grep, plus read-only access to the ledger and the
hypothesis list. It can read the record and the deliverable and it can write neither. The
implementer is the only node that calls the evaluator, through get_evaluator(), so every
evaluation is ledgered with provenance and the count cannot be inflated.

A node that cannot write the record cannot quietly repair a result it dislikes. That separation
is enforced by the tool set rather than by instruction.

## Specialization, one model per node

Different jobs, different models: a strong model can plan while a cheaper one executes.

Each node can take its own backend and its own settings, including a local open-weights model.

The slide promises more than the default ships.

The mechanism is real. _src/agent_runtime.py resolves a node's model as its own setting where one
is present and the run's otherwise, so any node can take a different model. But no shipped agent
in _src/agents/ sets one, which makes the default graph homogeneous: five nodes, one model. The
single mixed run used a stronger strategizer with cheaper workers and was configured by hand.

The open-weights parity claim is not benchmarked here. It is headroom rather than a result.

## Efficiency through parallel, cancellable work

Work is handed over as a **delegation**: a unit you can cancel, retry, or abandon. A six-hour run that dies at hour five does not start again.

Independent units also run at the same time.

<p align="center"><img src=../figures/hypothesis_delegation.png width=78%></p>


The _delegation_id column on the next code slide is this same object, stamped onto every row the
delegation produced.

The figure invites the reading that a delegation is a hand-off to another graph node. It is not.
Delegate() copies the target agent and runs it in a daemon thread inside the calling node.
Several run concurrently, and the planner is re-prompted as reports arrive. The only graph
transitions are the node returning to itself and the run reaching END.

Cancellable means the unit of work is abandoned and its record kept. A running solver is not
killed mid-solve.

## The shared state is a file on disk

A node is stateless between calls.

Agent frameworks usually checkpoint the whole conversation, so a thread can be resumed.
We made the record the state instead. That buys reproducibility rather than resumable
dialogue: every claim in the deliverable recomputes from the record.

The trade is explicit: resumable dialogue was given up. What it buys is that a run can be audited
by someone who never saw the conversation, because every number in the deliverable is recomputed
from experiment_data/ rather than quoted from a transcript.

A run closes only through an accepted Done(). Nine distinct paths can end a run, and 19.2 covers
them.

The limit is worth stating. Recomputing a number from the record checks the arithmetic and the
bookkeeping. It does not check the physics.

In [3]:
# A record from a real run: 538 evaluations of a lattice design.
ledger = ExperimentData.from_file('.')
design, result = ledger.to_pandas()
record = design.join(result)

# the design columns are your final project's variables; the _ columns are the stamps
record[["ratio_a", "ratio_pitch", "coilable", "sigma_crit",
        "_delegation_id", "_source", "_wall_ms"]].head(6)

,ratio_a,ratio_pitch,coilable,sigma_crit,_delegation_id,_source,_wall_ms
0,0.022644,0.680000,1,4.046822,D000,supercompressible-material,70499.483
1,0.009213,0.681277,1,0.807276,D000,supercompressible-material,239986.841
2,0.009213,0.681277,0,NaN,D000,supercompressible-material,142069.682
3,0.009213,0.681277,0,3.245657,D000,supercompressible-material,141805.785
4,0.026858,1.317206,0,21.218537,D000,supercompressible-material,141822.288
5,0.009213,0.681277,0,3.245657,D000,supercompressible-material,141473.844


Two columns not selected here hold absolute cluster paths, including a scratch directory and a
username. The seven columns shown carry no such paths.

The data is 538 evaluations from the supercompressible metamaterial study, the same problem as
19.2, which is what _source records. ratio_a and ratio_pitch are geometry ratios; coilable is the
screen for whether a design coils rather than collapsing locally.

Row 2 is a screened-out design. No solve was run for it, which is why its outputs are empty
rather than zero.

## Choosing your own topology

Ours has one node that plans and delegates to the others. That is an accident of our problem,
not a principle: it is a bottleneck and a single point of failure.

Your problem, your graph.

The planner the slide leaves anonymous is the strategizer, which is also the graph's entry node.
The five in the ratified topology are the strategizer, the literature reviewer, the data
generator, the implementer and the critic.

A hub is both a bottleneck and a single point of failure: every result travels through the
strategizer to reach any other node.

The change we would make is to give the workers edges to each other, so a result need not be
relayed by the planner to be used. The hub is not a principle we would defend.

## Popper's advice

As of 2026, LLMs are generalists by default.

They produce reasonable-sounding answers, regress to the mean, and are probabilistic throughout.

We push back by compiling the rules of science into every agent that judges a claim. We start from Popper.


The rules are not requested politely at runtime. FALSIFICATION_CHARTER is compiled verbatim into
the system prompt of every node that judges a claim, the strategizer and the critic, identically
and at class definition. Because both receive the same numbered text, either can cite a clause by
number and the other defers to the same words, so there is no paraphrase drift and no negotiation
over what falsification means.

Being a stable prefix, it is cached by the SDK and costs almost nothing per turn.

Popper's account is contested. What was needed here is a rule that stops a model confirming
itself.

### Science rule 1: one falsifiable claim

One file, quoted verbatim into every node that judges a claim. §1:

> A hypothesis is ONE falsifiable claim carrying a registered prediction: the observable
> whose occurrence would refute the claim.

Register what would refute you, before you look.


The file is FALSIFICATION_CHARTER in a3dasm/_src/knowledge/charter.py: six numbered clauses,
compiled into the strategizer's and the critic's prompts, and reachable by workers on demand
through ConsultHandbook.

Two of the six are quoted in this lecture. This is the definition; the next slide has the clause
on corroboration.

Register means written down before the data is seen. The critic enforces it: a hypothesis with no
registered prediction receives no closing verdict. A further clause requires that a FALSIFIED
verdict rest on the same prediction that was registered, not on an observation chosen after
seeing the data.

### Science rule 2: corroboration, not proof

§5:

> SUPPORTED [...] means the hypothesis survived at least one adequate attempt to refute
> it. You never "confirm" a hypothesis; you only fail to falsify it.

Four statuses, and only four: `OPEN`, `SUPPORTED`, `FALSIFIED`, `INCONCLUSIVE`.


The four statuses are the only ones. OPEN means no adequate test yet. The other three are closing
statuses, and each must cite a real delegation together with a concrete result that bears on the
registered prediction.

FALSIFIED means an adequate test contradicted that prediction. INCONCLUSIVE means the test was
not adequate, so the claim survives untested. A contradiction produced by a flawed test indicts
the test rather than the claim, which is the Duhem-Quine point and is stated explicitly in the
charter.

SUPPORTED is corroboration, not proof: the claim survived one adequate attempt to refute it.

## The reproduction gate

The deliverable is a notebook. Its code cells recompute the result from the record.

It is executed in a clean sandbox before the run may close.


The deliverable gets six attempts to reproduce itself. If it still will not run, the run is
stamped FAILED and the critic is never spent on it.

Clean means a fresh process that never saw the run: no variables in memory, no cached state, only
what is on disk. That is what makes the notebook evidence rather than a transcript.

The limit is named in the project's own backlog. Reproduction is enforced hard while scientific
adequacy is enforced by judgement. Re-running the notebook checks the arithmetic; the charter
and the critic are what address whether the physics was right.

## The study folder

```
my_study/
  PROBLEM_STATEMENT.md   # required: the brief
  config.yaml            # optional: model, budget, how a design is scored
  workspace/
    evaluator.py         # optional: your ground truth
```

Only the problem statement is required. Everything else has a default: no config file means the
default model and no budget, and no evaluator means the system must construct one, which is the
data generator's job and the reason that node exists.

Results land in experiment_data/ under the study directory, the folder loaded two slides ago.

In the 19.2 campaign the problem statement is itself reviewed before any work starts, against
five elements: objective, design space, ground truth, validity and deliverable. That review is
written to debug/problem_statement_review.md, and it is present in all 36 runs.

## The problem statement file

```markdown
# Minimise a 2-D quadratic

## Objective
Minimise y = (x1 - 1)^2 + (x2 + 2)^2.

## Design space
| variable | type | bounds | units |
|---|---|---|---|
| x1 | continuous | [-5, 5] | dimensionless |
| x2 | continuous | [-5, 5] | dimensionless |
```

Objective, bounds, units. That table is a `Domain`, written in prose.

## Model, budget, and evaluator

```yaml
model: haiku
eval_budget: 200
evaluator:
  entrypoint: "workspace/evaluator.py:evaluate"
  output_names: [y]
```

```python
def evaluate(x1: float, x2: float) -> float:
    return (x1 - 1.0) ** 2 + (x2 + 2.0) ** 2
```

The model string is a pinned identifier rather than a friendly name. The default is
claude-haiku-4-5-20251001, set at a3dasm/_src/agent_runtime.py:43. Resolution runs explicit
argument, then this file, then a backend default, which is qwen2.5:1.5b against a local ollama
server.

An alias table exists, but only on the open-weights path: MODEL_ALIASES at slurm_llm.py:104
expands gemma-4 to a full HuggingFace id, and resolve_model_id is called once, at
slurm_llm.py:560. A short name such as haiku is not expanded on the Anthropic path. It is passed
through verbatim and fails.

The 19.2 study overrides this in config.yaml with model: claude-sonnet-5. eval_budget is
advisory: it warns, and it never stops a run.

## One call, one notebook

```python
from a3dasm import AgenticRun
report = AgenticRun(study_dir="my_study").execute()
```

Back comes `pipeline.ipynb`, the record of every evaluation, and whether the run passed
its gate.

Costs belong to 19.2, where one real run is 8.8 hours and about $28 in model calls.

Credentials are needed for whichever backend is chosen. Four are registered and two of them are
local, so an ollama or vLLM server on your own hardware is a supported path rather than a
workaround.

The hard stops are a spend ceiling, twelve consecutive errors from a single target, and a time
backstop. The budgets are not hard stops.

## An agent's account of a failure

Every node writes a retrospective when a run closes: what it found hard, what blocked it, where it
contradicted itself. It is the highest-signal artifact we have, and it can still be wrong about
mechanism.

One run's retrospective blamed a silent crash for a 63% evaluation failure rate, and pointed
at the licence server saturating at 16-way concurrency.

The 63% is a quotation from the run's own retrospective rather than a finding. It cannot be
reproduced from the record, and the denominator it was measured over is unknown.

Retrospectives are first-person entries a node writes as the run closes, in
debug/retrospectives.jsonl, under four headings: CONSISTENCY, DECISION, FRICTION and BLOCKED.
They are the highest-signal artifact for what went wrong, and a FRICTION entry often names a root
cause that appears in no traceback.

This one was wrong about the mechanism and about the rate. It is a lead rather than evidence,
which the next slide shows.

In [4]:
# What the run reported, against what the record shows.
d = design.join(result)
attempted = d[d.coilable == 1]              # passed the coilability screen
failed = ~attempted.riks_converged.astype(bool)

print("solves that never converged:", int(failed.sum()), "of", len(attempted))
print("failure rate:", round(100 * failed.mean(), 1), "%")
print("median wall time, failed vs converged:",
      int(attempted.loc[failed, '_wall_ms'].median() / 1000), "s vs",
      int(attempted.loc[~failed, '_wall_ms'].median() / 1000), "s")

solves that never converged: 154 of 469
failure rate: 32.8 %
median wall time, failed vs converged: 329 s vs 66 s


The retrospective is a lead. The record is the diagnosis: these solves ran five times longer than
the ones that worked before they died, which is a wait, not a crash. The 63% is not in the record either: the rate here is 32.8%.

The numbers on screen: 154 of 469 attempted solves never converged, and the failures ran a median
329 seconds against 66 seconds for those that succeeded.

The inference that needs defending is the last one. A long run that then dies is consistent with a
queue or a licence wait, whereas a crash is short and then dead. That is an argument from timing
rather than a measurement of the licence server, and the difference is worth conceding.

The remaining 69 of the 538 rows never passed the coilability screen, so no solve was attempted
for them.

## Summary

* An agentic workflow is a graph of nodes, each a model with tools, choosing its own next step.
* Split the work for **safety**, **specialization** and **efficiency**, and not otherwise.
* The state is the data, not the conversation.
* A claim is admitted by surviving refutation and by reproducing from the record.
* One folder in, one notebook out.

What we would build differently is the hub. Every result travels
through the strategizer, which makes it both the bottleneck and the single point of failure, and
the workers should have had edges to each other.

19.2 applies the same machinery to a real metamaterial design problem. It closes on two design
studies in which the mechanism a design was supposed to exhibit turned out not to be the one
producing the number, once where the design failed and once where it won.

## Next lecture

The same framework, pointed at a real design problem, and what it actually found.

### See you next class

Have fun!